# 07 · Human-in-the-Loop: Pause, Approve, Edit, Resume

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama.

Some decisions shouldn't be left to a model: sending an email, spending money,
publishing, deleting. LangGraph lets a graph **pause** at any point, hand a
question to a human, and **resume** when the answer arrives, whether that's in
seconds or days. The process doesn't have to stay alive while it waits.

**You will learn**
- `interrupt()`: how a node pauses, and why a **checkpointer** and **thread id** are required
- Resuming with `Command(resume=...)`, and the rule that **the node re-runs from the top**
- Four patterns: **approve/reject**, **edit state**, **review tool calls**, **ask for input**
- Validating human input, and handling **several interrupts at once**
- `HumanInTheLoopMiddleware`: approval gates for `create_agent` in a few lines

**Prerequisites:** notebooks 04 (tools) and 05 (`Command`).
*Optional background:* `00_CONCEPTS/01` §3.4 shows why a blocking approval
callback can't wait for a human.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

from typing import Literal, TypedDict, Annotated
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode
from common import get_model

model = get_model()

## 1 · Why: a pause, not a block

The naive way to ask a human is a callback: `if not input('approve?'): ...`. That
**blocks**. The program sits holding everything in memory until someone
answers. Real approvals take hours, and the process will be restarted long
before then.

LangGraph's approach:

```
 invoke ──► node A ──► node B calls interrupt(question)
                          │
                          ▼
          save checkpoint (state + "B is waiting" + the question)   ← durable
          invoke() RETURNS, with __interrupt__ = [question]           ← nobody is blocked
                         ...  minutes, hours, days  ...
 invoke(Command(resume=answer), same thread_id)
          load checkpoint ──► re-run node B from the top; interrupt() now RETURNS answer
                          ──► continue the graph
```

Pausing therefore requires two things:
- a **checkpointer**: where the paused state is saved (in memory here; SQLite in notebook 08)
- a **thread id**: which saved run to resume, passed as `config={'configurable': {'thread_id': ...}}`

## 2 · The basics: approve a draft before publishing

In [ ]:
class Post(TypedDict):
    topic: str
    draft: str
    decision: str
    status: str


def write(state: Post) -> dict:
    text = model.invoke(f"Write a one-sentence LinkedIn post about {state['topic']}. Sentence only.").content
    return {'draft': text}


def human_approval(state: Post) -> Command[Literal['publish', 'discard']]:
    answer = interrupt({'question': 'Publish this post?', 'draft': state['draft']})   # ← pauses here
    goto = 'publish' if answer == 'yes' else 'discard'
    return Command(update={'decision': answer}, goto=goto)


def publish(state: Post) -> dict:
    return {'status': 'published ✅'}

def discard(state: Post) -> dict:
    return {'status': 'discarded 🗑️'}


b = StateGraph(Post)
for fn in (write, human_approval, publish, discard):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'write')
b.add_edge('write', 'human_approval')
b.add_edge('publish', END)
b.add_edge('discard', END)

memory = InMemorySaver()
poster = b.compile(checkpointer=memory)          # ← a checkpointer is required to resume

config = {'configurable': {'thread_id': 'post-1'}}
result = poster.invoke({'topic': 'why robots need battery redundancy'}, config)

print('invoke returned. Keys:', list(result))
question = result['__interrupt__'][0]
print('paused with   :', question.value)

`invoke` **returned**. Nothing is blocked, and the paused run lives in the
checkpointer under `thread_id='post-1'`. Any code (a web handler, a Slack bot,
a cron job) can inspect it later:

In [ ]:
snapshot = poster.get_state(config)
print('waiting at :', snapshot.next)
print('asking     :', snapshot.interrupts[0].value['question'])
print('state      :', {k: v for k, v in snapshot.values.items() if k != 'draft'})

When the human answers, resume **the same thread** with `Command(resume=...)`.
The value you pass becomes the return value of `interrupt()` inside the node:

In [ ]:
final = poster.invoke(Command(resume='yes'), config)
print(final['status'], '| decision:', final['decision'])

## 3 · The rule you must remember: the node re-runs from the top

On resume, LangGraph doesn't continue from the middle of the function. It
**runs the interrupted node again from its first line**, and this time
`interrupt()` returns the answer instead of pausing. Anything *before*
`interrupt()` therefore runs **twice**:

In [ ]:
side_effects = []

class Tiny(TypedDict):
    answer: str

def risky_node(state: Tiny) -> dict:
    side_effects.append('email sent (BEFORE interrupt)')        # ✗ runs on pause AND on resume
    answer = interrupt('Continue?')
    side_effects.append('email sent (AFTER interrupt)')         # ✓ runs once, after the answer
    return {'answer': answer}

b = StateGraph(Tiny)
b.add_node('risky_node', risky_node)
b.add_edge(START, 'risky_node')
g = b.compile(checkpointer=InMemorySaver())
cfg = {'configurable': {'thread_id': 'demo'}}

g.invoke({'answer': ''}, cfg)
g.invoke(Command(resume='ok'), cfg)
for s in side_effects:
    print(s)

**Rules that follow:**
- Put side effects (emails, payments, API writes) **after** `interrupt()`, or in a later node.
- Expensive work before `interrupt()` (like an LLM call) is repeated on resume.
  Do it in an *earlier* node (as `write` does above) so its result is checkpointed.
- If a node has several `interrupt()` calls, they're matched to resume values
  **by order**, so don't put them inside conditions that could change between runs.

## 4 · Pattern: the human edits the state

Approval is binary. Often the human wants to **fix** the draft instead. The
resume value can be anything JSON-like, so send a small command object:

In [ ]:
def human_edit(state: Post) -> Command[Literal['publish', 'discard']]:
    reply = interrupt({'draft': state['draft'], 'options': ['approve', 'edit', 'reject']})
    if reply['action'] == 'edit':
        return Command(update={'draft': reply['text'], 'decision': 'edited'}, goto='publish')
    if reply['action'] == 'approve':
        return Command(update={'decision': 'approved'}, goto='publish')
    return Command(update={'decision': 'rejected'}, goto='discard')


b = StateGraph(Post)
for fn in (write, human_edit, publish, discard):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'write')
b.add_edge('write', 'human_edit')
b.add_edge('publish', END)
b.add_edge('discard', END)
editor = b.compile(checkpointer=InMemorySaver())

cfg = {'configurable': {'thread_id': 'post-2'}}
paused = editor.invoke({'topic': 'sidewalk robots in the rain'}, cfg)
print('model draft :', paused['__interrupt__'][0].value['draft'])

done = editor.invoke(Command(resume={'action': 'edit',
                                     'text': 'Our robots now handle rain safely. Details on our blog.'}), cfg)
print('published   :', done['draft'], '|', done['status'])

## 5 · Pattern: review tool calls before they run

The most important use of human-in-the-loop in agents: the model may *request*
a risky tool, but a human decides whether it *executes*. We insert a review node
between the model and the tools:

```
 START → agent ──(tool calls?)──► review ──approve/edit──► tools ──► agent
            ▲                        │
            └──── reject (a ToolMessage saying "rejected") ◄┘
```

- **approve**: go to `tools` unchanged.
- **edit**: replace the AI message (same `id`, so `add_messages` **replaces** it)
  with corrected arguments, then run the tools.
- **reject**: answer each tool call with a `ToolMessage` saying it was rejected, and
  go back to the agent, which must now respond without it.

In [ ]:
outbox = []                                      # what REALLY got sent

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    outbox.append({'to': to, 'subject': subject})
    return f'Email sent to {to}.'


agent_llm = model.bind_tools([send_email])

def agent(state: MessagesState) -> dict:
    return {'messages': [agent_llm.invoke(state['messages'])]}


def review(state: MessagesState) -> Command[Literal['tools', 'agent']]:
    request = state['messages'][-1]
    reply = interrupt({'tool_calls': request.tool_calls})            # show the human what WOULD run
    if reply['action'] == 'approve':
        return Command(goto='tools')
    if reply['action'] == 'edit':
        edited = AIMessage(content=request.content, id=request.id,    # same id → replaces the request
                           tool_calls=[{**tc, 'args': {**tc['args'], **reply['args']}}
                                       for tc in request.tool_calls])
        return Command(update={'messages': [edited]}, goto='tools')
    rejected = [ToolMessage(f"Rejected by a human: {reply.get('reason', '')}", tool_call_id=tc['id'])
                for tc in request.tool_calls]
    return Command(update={'messages': rejected}, goto='agent')


def route(state: MessagesState) -> Literal['review', '__end__']:
    return 'review' if state['messages'][-1].tool_calls else END


b = StateGraph(MessagesState)
b.add_node('agent', agent)
b.add_node('review', review)
b.add_node('tools', ToolNode([send_email]))
b.add_edge(START, 'agent')
b.add_conditional_edges('agent', route)
b.add_edge('tools', 'agent')
mailer = b.compile(checkpointer=InMemorySaver())
print(mailer.get_graph().draw_ascii())

In [ ]:
def ask(thread, text):
    cfg = {'configurable': {'thread_id': thread}}
    out = mailer.invoke({'messages': [('user', text)]}, cfg)
    for call in out['__interrupt__'][0].value['tool_calls']:
        print('  model wants :', call['name'], call['args'])
    return cfg

def resume(cfg, decision):
    out = mailer.invoke(Command(resume=decision), cfg)
    print('  agent says  :', out['messages'][-1].content[:150])


print('A) approve')
cfg = ask('mail-a', 'Email sam@example.com that the robot demo moved to 3pm.')
resume(cfg, {'action': 'approve'})

print('\nB) edit the recipient')
cfg = ask('mail-b', 'Email sam@example.com that the robot demo moved to 3pm.')
resume(cfg, {'action': 'edit', 'args': {'to': 'team@example.com'}})

print('\nC) reject')
cfg = ask('mail-c', 'Email the whole company that we are cancelling the Q3 rollout.')
resume(cfg, {'action': 'reject', 'reason': 'not approved by management'})

print('\noutbox (what actually got sent):', outbox)

The outbox is the proof: the approved email went out, the edited one went to
the corrected recipient, and the rejected one **never executed**. The model only
ever *requested* it.

## 6 · Pattern: ask the human for missing information

A tool can itself be a question to a human. When the model calls `ask_user`, the
graph pauses, and the human's reply becomes the tool's result:

In [ ]:
@tool
def ask_user(question: str) -> str:
    """Ask the user a clarifying question when required information is missing."""
    return interrupt({'question': question})        # the human's answer IS the tool result

@tool
def book_room(room: str, time: str) -> str:
    """Book a meeting room at a time."""
    return f'Booked {room} at {time}.'


llm2 = model.bind_tools([ask_user, book_room])
b = StateGraph(MessagesState)
b.add_node('agent', lambda s: {'messages': [llm2.invoke(
    [('system', 'Book rooms. If the time or room is missing, call ask_user. Never guess.')] + s['messages'])]})
b.add_node('tools', ToolNode([ask_user, book_room]))
b.add_edge(START, 'agent')
b.add_conditional_edges('agent', lambda s: 'tools' if s['messages'][-1].tool_calls else END, ['tools', END])
b.add_edge('tools', 'agent')
booker = b.compile(checkpointer=InMemorySaver())

cfg = {'configurable': {'thread_id': 'booking'}}
out = booker.invoke({'messages': [('user', 'Book the Everest room for the design review.')]}, cfg)
print('agent asks :', out['__interrupt__'][0].value['question'])
out = booker.invoke(Command(resume='Tomorrow at 10:00'), cfg)
print('agent says :', out['messages'][-1].content)

## 7 · Validating human input, and several interrupts at once

**Validation.** Humans make typos. Because `interrupt()` can be called in a
loop, a node can keep asking until the answer is valid. Each resume answers the
next `interrupt()` in order:

In [ ]:
class Budget(TypedDict):
    amount: int

def ask_budget(state: Budget) -> dict:
    prompt = 'Budget in EUR (a positive whole number)?'
    while True:
        raw = interrupt(prompt)
        if str(raw).isdigit() and int(raw) > 0:
            return {'amount': int(raw)}
        prompt = f'{raw!r} is not valid. Budget in EUR (a positive whole number)?'

b = StateGraph(Budget)
b.add_node('ask_budget', ask_budget)
b.add_edge(START, 'ask_budget')
g = b.compile(checkpointer=InMemorySaver())
cfg = {'configurable': {'thread_id': 'budget'}}

out = g.invoke({'amount': 0}, cfg)
for answer in ['lots', '-5', '2500']:
    print('asked:', out['__interrupt__'][0].value, '→ answering', repr(answer))
    out = g.invoke(Command(resume=answer), cfg)
print('stored amount:', out['amount'])

**Parallel interrupts.** If two parallel branches both interrupt in the same
superstep, `invoke` returns **both** interrupts. Resume them together with a
dict mapping each interrupt's `id` to its answer:

In [ ]:
import operator

class Sign(TypedDict):
    signatures: Annotated[list[str], operator.add]

def legal(state):   return {'signatures': [f"legal: {interrupt('Legal sign-off?')}"]}
def finance(state): return {'signatures': [f"finance: {interrupt('Finance sign-off?')}"]}

b = StateGraph(Sign)
b.add_node('legal', legal)
b.add_node('finance', finance)
b.add_edge(START, 'legal')
b.add_edge(START, 'finance')
g = b.compile(checkpointer=InMemorySaver())
cfg = {'configurable': {'thread_id': 'signoff'}}

pending = g.invoke({'signatures': []}, cfg)['__interrupt__']
for i in pending:
    print('pending:', i.value, '(id', i.id[:8] + '…)')
answers = {i.id: ('approved' if 'Legal' in i.value else 'approved with conditions') for i in pending}
print(g.invoke(Command(resume=answers), cfg)['signatures'])

## 8 · The shortcut: `HumanInTheLoopMiddleware` for `create_agent`

The tool-review graph in §5 is common enough to be prebuilt. With
`create_agent`, add the middleware and say which tools need approval:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def read_inbox() -> str:
    """Read the latest emails."""
    return 'From Sam: can we move the robot demo to 3pm?'

outbox.clear()
assistant = create_agent(
    model,
    tools=[read_inbox, send_email],
    system_prompt='You are an email assistant.',
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        'send_email': True,         # pause: approve, edit, reject or respond
        'read_inbox': False,        # safe: runs without asking
    })],
    checkpointer=InMemorySaver(),
)

cfg = {'configurable': {'thread_id': 'assistant-1'}}
out = assistant.invoke({'messages': [('user', 'Read my inbox and reply to Sam that 3pm works.')]}, cfg)
request = out['__interrupt__'][0].value
for action in request['action_requests']:
    print('needs approval:', action['name'], action['args'])

out = assistant.invoke(Command(resume={'decisions': [{'type': 'approve'}]}), cfg)
print('agent says:', out['messages'][-1].content[:150])
print('outbox    :', outbox)

Look at the arguments before approving. In one of our runs the model addressed
the email to plain `sam` instead of an email address: exactly the kind of
mistake an approval gate exists to catch (an `edit` decision fixes it).

The resume value is `{'decisions': [...]}` with one decision per pending tool call:

| Decision | Shape | Effect |
|---|---|---|
| approve | `{'type': 'approve'}` | run the tool as requested |
| edit | `{'type': 'edit', 'edited_action': {'name': ..., 'args': {...}}}` | run it with your changes |
| reject | `{'type': 'reject', 'message': 'why'}` | don't run it; the model is told why |
| respond | `{'type': 'respond', 'message': '...'}` | don't run it; your text becomes the tool's result |

**Static breakpoints** (`compile(interrupt_before=['tools'])`) also exist. They
pause before a node *every time*, which is handy for debugging, but for
approvals `interrupt()` is better: it carries a payload, can be conditional,
and lives in your code.

## 9 · Trade-offs and design guidance

- **Put the pause where the risk is.** Gate side-effecting tools, not every step.
  Too many approvals and humans start clicking "yes" without reading.
- **Show the human exactly what will happen:** the tool name and full arguments,
  or the final draft, never a summary.
- **Make resumes idempotent.** A human may click "approve" twice; the second
  resume should be harmless (the thread is no longer waiting).
- **Plan for timeouts.** Paused threads wait forever. Decide what happens to
  runs nobody answers (expire, escalate, auto-reject).
- **In production, use a durable checkpointer.** `InMemorySaver` loses every
  paused run when the process restarts. Notebook 08 switches to SQLite.

## 10 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| No checkpointer | the run pauses but can never be resumed | `compile(checkpointer=...)` |
| Different `thread_id` on resume | resume starts a new run instead | store and reuse the thread id |
| Side effects before `interrupt()` | they happen twice | move them after, or to a later node |
| LLM call before `interrupt()` in the same node | paid twice, may differ on resume | do it in an earlier node |
| `interrupt()` inside a changing condition | answers matched to the wrong question | keep interrupt order deterministic |
| Rejecting without a `ToolMessage` | malformed history; the model is confused | answer every rejected tool call |

## 11 · Check yourself

<details><summary><b>1.</b> What two things does a graph need before <code>interrupt()</code> can be resumed, and why?</summary>

A checkpointer (to save the paused state, including which node is waiting and what it asked) and a thread id (to identify which saved run to load and continue).
</details>

<details><summary><b>2.</b> A node calls an LLM, then <code>interrupt()</code>, then sends an email. What happens on resume, and how should you restructure it?</summary>

The node re-runs from the top: the LLM call is repeated (cost, and possibly a different result the human never saw). The email is sent once, after the answer. Move the LLM call to an earlier node so its output is checkpointed and the human approves exactly what will be used.
</details>

<details><summary><b>3.</b> How does the "edit" branch in §5 replace the model's tool call instead of adding a second one?</summary>

It returns an `AIMessage` with the same `id` as the original request. The `add_messages` reducer replaces messages with matching ids.
</details>

<details><summary><b>4.</b> Why must a rejection add a <code>ToolMessage</code> for each tool call?</summary>

Every tool call in history needs a matching result, or the conversation is malformed. The rejection message also tells the model what happened, so it can respond appropriately instead of retrying blindly.
</details>

<details><summary><b>5.</b> Two parallel branches interrupt at once. How do you resume?</summary>

`invoke` returns both interrupts, each with an `id`. Resume with `Command(resume={id1: answer1, id2: answer2})`.
</details>

<details><summary><b>6.</b> When would you use <code>interrupt_before</code> instead of <code>interrupt()</code>?</summary>

For debugging or step-through inspection, where you always want to stop before a given node. For real approvals use `interrupt()`, because it can be conditional, carries a payload for the human, and returns their answer.
</details>

## Takeaway

`interrupt()` turns "wait for a human" into **save and return**: the graph
checkpoints, `invoke` returns the question, and `Command(resume=...)` on the same
thread continues it later, re-running the paused node from the top. Use it to
approve, edit, or reject actions, and above all to review tool calls before
they execute. Keep side effects after the interrupt.

Next → `08_persistence_and_time_travel.ipynb`: durable SQLite checkpoints,
conversations that remember, surviving crashes, and replaying history.